In [1]:
from utils import simulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import os
import hashlib
import psutil
import time

### Simulations Parameter ##############
seed = 42
n_sim = 1_0

B_RF = 100
B_1  = 200

jk_ab_calc = True
boot_var_calc = False
ijk_std_calc = True
train_models = True

# Funktion zur Überwachung der Ressourcennutzung
def monitor_resources():
    cpu_usage = psutil.cpu_percent(interval=1)
    memory_info = psutil.virtual_memory()
    print(f"CPU usage: {cpu_usage}%")
    print(f"Memory usage: {memory_info.percent}%")


for n_i in [100]:
    for _ in [1]:
        ########## Varying Parameters ##########

        n = 750  #n_i

        tau = 37

        params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 24_300, 'rate_censoring': 0.003,
                                'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                                'n': n, 'seed': seed, 'tau': tau}


        params_rf = {  'n_estimators':B_RF,                        
                        'max_depth':3,
                        'min_samples_split':5,
                        'max_features': 'log2',
                        'random_state':  seed,
                        'weighted_bootstrapping':     True,  }

        X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})

        ######## Start Simulation   ########
        with ProcessPoolExecutor() as executor:
            
            ### Array to store the results
            portion_events_after_cut_train = np.zeros(n_sim)
            portion_censored_after_cut_train = np.zeros(n_sim)
            portion_no_events_after_cut_train = np.zeros(n_sim)
            portion_events_after_cut_test = np.zeros(n_sim)
            portion_censored_after_cut_test = np.zeros(n_sim)
            portion_no_events_after_cut_test = np.zeros(n_sim)
            wb_mse_ipcw = np.zeros(n_sim)
            wb_cindex_ipcw = np.zeros(n_sim)
            wb_y_pred_X_point = np.zeros(n_sim)
            rf_mse_ipcw = np.zeros(n_sim)
            rf_y_pred_X_point = np.zeros(n_sim)
            ijk_var_pred_X_point = np.zeros(n_sim)
            bootstrap_var_pred_X_point = np.zeros(n_sim)
            jk_ab_var_pred_X_point = np.zeros(n_sim)
            

            futures = [
                executor.submit(
                    simulation,
                    seed=seed+i,
                    tau=tau, 
                    data_generation_weibull_parameters=params_data_creation,
                    params_rf=params_rf, 
                    X_pred_point=X_erwartung,
                    B_first_level = B_1,
                    boot_std_calc = boot_var_calc,
                    ijk_std_calc = ijk_std_calc,
                    train_models = train_models,
                    jk_ab_calc = jk_ab_calc
                )
                for i in range(n_sim)
            ]

            for i, future in enumerate(tqdm(futures, desc="Simulations", unit="simulation")):
                _portion_events_after_cut_train, _portion_censored_after_cut_train, _portion_no_events_after_cut_train, \
                _portion_events_after_cut_test, _portion_censored_after_cut_test, _portion_no_events_after_cut_test, \
                _wb_mse_ipcw, _wb_cindex_ipcw, _wb_y_pred_X_point, \
                _rf_mse_ipcw, _rf_y_pred_X_point, _ijk_var_pred_X_point, _bootstrap_var_pred_X_point, _jk_ab_var_pred_X_point  = future.result()

                if i<3:
                    # Überwache Ressourcen nach jeder Simulation
                    monitor_resources()

                #Event-Stats Results
                portion_events_after_cut_train[i] = _portion_events_after_cut_train
                portion_censored_after_cut_train[i] = _portion_censored_after_cut_train
                portion_no_events_after_cut_train[i] = _portion_no_events_after_cut_train
                portion_events_after_cut_test[i] = _portion_events_after_cut_test
                portion_censored_after_cut_test[i] = _portion_censored_after_cut_test
                portion_no_events_after_cut_test[i] = _portion_no_events_after_cut_test
                
                #Evaluation Results
                wb_mse_ipcw[i] = _wb_mse_ipcw
                wb_cindex_ipcw[i] = _wb_cindex_ipcw
                rf_mse_ipcw[i] = _rf_mse_ipcw

                #Prediction Results
                wb_y_pred_X_point[i] = _wb_y_pred_X_point[0]
                rf_y_pred_X_point[i] = _rf_y_pred_X_point[0]

                # Standard Deviation Estimates
                ijk_var_pred_X_point[i]  = _ijk_var_pred_X_point[0]
                jk_ab_var_pred_X_point[i] = _jk_ab_var_pred_X_point
                bootstrap_var_pred_X_point[i] = _bootstrap_var_pred_X_point

        ########## Save Results ############
        results = pd.DataFrame({'wb_mse_ipcw': wb_mse_ipcw,
                                'wb_cindex_ipcw': wb_cindex_ipcw,
                                'rf_mse_ipcw': rf_mse_ipcw,
                                'wb_y_pred_X_point': wb_y_pred_X_point,
                                'rf_y_pred_X_point': rf_y_pred_X_point,
                                'ijk_var_pred_X_point': ijk_var_pred_X_point,
                                'jk_ab_var_pred_X_point': jk_ab_var_pred_X_point,
                                'bootstrap_var_pred_X_point': bootstrap_var_pred_X_point})

        path = os.path.abspath('')
        experiment_name = f'n_train {n*0.7}_Events {round(n*0.7*np.mean(portion_events_after_cut_train),0)} ({round(np.mean(portion_events_after_cut_train)*100,2)} %)_Cens {round(n*0.7*np.mean(portion_censored_after_cut_train),0)} ({round(np.mean(portion_censored_after_cut_train)*100,2)} %)_(B_RF) {B_RF}__(B_1) {B_1}'

        if not os.path.exists(path + '/results/'+experiment_name):
            os.makedirs(path + '/results/'+experiment_name)

        ## Abspeichern der Ergebnise in einer .csv Datei und txt Datei in results ordner
        results.to_csv(path + '/results/'+experiment_name+'/'+experiment_name+'.csv', index=False)
        if not os.path.exists(path + '/results_txt'):
            os.makedirs(path + '/results_txt')
        with open(path + '/results/'+experiment_name+'/results.txt', 'w') as f:

            f.write(f'n_train= {n*0.7}// Events: {round(n*0.7*np.mean(portion_events_after_cut_train),0)} ({round(np.mean(portion_events_after_cut_train)*100,2)} %) // Censored: {round(n*0.7*np.mean(portion_censored_after_cut_train),0)} ({round(np.mean(portion_censored_after_cut_train)*100,2)} %) // B_RF: {B_RF} // (B_1): {B_1} \n')

            f.write('\n### Standard Deviation: ###\n')
            a = np.std(wb_y_pred_X_point,ddof=1)
            f.write(f'WB EMP-STD:                 {a:.4f}\n')
            b = np.std(rf_y_pred_X_point,ddof=1)
            f.write(f'RF EMP-STD:                 {b:.4f}\n\n')

            non_neg_var_ijk_est = ijk_var_pred_X_point.copy()
            non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
            f.write(f'IJK STD (for RF) Mean-est               : {a:.4f}  \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} % \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.2f}\n\n')
            
            non_neg_var_ijk_est = jk_ab_var_pred_X_point.copy()
            non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
            f.write(f'JK-AB(un-weighted) STD (for RF) Mean-est: {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.2f} \n\n')

            a = np.mean(np.sqrt(  bootstrap_var_pred_X_point    ))
            f.write(f'Boot STD (for RF) Mean-est              : {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  a    )):.2f} \n\n')




            f.write('\n\n### mean Data Stats over all simulations: ###\n')
            f.write('Number of simulations: '+str(n_sim)+'\n')
            f.write('cut-off time tau: '+str(tau)+'\n\n')
            f.write(f'Train ({n*0.7}):\n')
            f.write(f'Events:     {round(np.mean(portion_events_after_cut_train)*100,2)}  %,  n={round(n*0.7*np.mean(portion_events_after_cut_train),0)}\n')
            f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_no_events_after_cut_train),0)}\n')
            f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_censored_after_cut_train),0)}\n')
            f.write(f'Test  ({n*0.3}):\n')
            f.write(f'Events:     {round(np.mean(portion_events_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_events_after_cut_test),0)}\n')
            f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_test)*100,2)} %,   n={round(n*0.3*np.mean(portion_no_events_after_cut_test),0)}\n')
            f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_censored_after_cut_test),0)}\n')


            f.write('\n\n### Evaluation: ###\n')
            f.write(f'WB C-Index IPCW: {np.mean(wb_cindex_ipcw):.4f}\n')
            f.write(f'WB MSE IPCW: {np.mean(wb_mse_ipcw):.4f}\n')
            f.write(f'RF MSE IPCW: {np.mean(rf_mse_ipcw):.4f}\n')
            f.write('\n')

            f.write('\n###Prediction Results:###\n')
            f.write(f'True Y: {99999999999999:.4f}\n')
            f.write(f'WB Y_pred: {np.mean(wb_y_pred_X_point):.4f}\n')
            f.write(f'RF Y_pred: {np.mean(rf_y_pred_X_point):.4f}\n')
            f.write('\n')



            f.write('\n\n### Simulation Parameters: ###\n')
            f.write('first_level_B for bootstrap variance estimation (B_1): '+str(B_1)+'\n')

            f.write('Data Creation Parameters:\n')
            f.write(str(params_data_creation))
            
            f.write('\nRandom Forest Parameter:\n')
            f.write(str(params_rf)+'\n')
            f.write(f'the above seeds ({seed}) are start_seed for the simulation function\n')

        ## Abspeichern der Ergebnise in einer .txt Datei in results_txt ordner
        hash_object = hashlib.md5(experiment_name.encode())
        experiment_hash = hash_object.hexdigest()
        with open(path + '/results_txt/'+experiment_hash+'.txt', 'w') as f:

            f.write(f'n_train= {n*0.7}// Events: {round(n*0.7*np.mean(portion_events_after_cut_train),0)} ({round(np.mean(portion_events_after_cut_train)*100,2)} %) // Censored: {round(n*0.7*np.mean(portion_censored_after_cut_train),0)} ({round(np.mean(portion_censored_after_cut_train)*100,2)} %) // B_RF: {B_RF} // (B_1): {B_1} \n')


            f.write('\n### Standard Deviation: ###\n')
            a = np.std(wb_y_pred_X_point,ddof=1)
            f.write(f'WB EMP-STD:                 {a:.4f}\n')
            b = np.std(rf_y_pred_X_point,ddof=1)
            f.write(f'RF EMP-STD:                 {b:.4f}\n\n')

            non_neg_var_ijk_est = ijk_var_pred_X_point.copy()
            non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
            f.write(f'IJK STD (for RF) Mean-est               : {a:.4f}  \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.2f} \n\n')
            
            non_neg_var_ijk_est = jk_ab_var_pred_X_point.copy()
            non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
            # erst die wurzel ziehen und dann den mittelwert bilden
            a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
            f.write(f'JK-AB(un-weighted) STD (for RF) Mean-est: {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.2f} \n\n')

            a = np.mean(np.sqrt(  bootstrap_var_pred_X_point    ))
            f.write(f'Boot STD (for RF) Mean-est              : {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.2f} %  \n std. des schätzers {np.std(np.sqrt(  a    )):.2f} \n\n')

            f.write('\n\n### mean Data Stats over all simulations: ###\n')
            f.write('Number of simulations: '+str(n_sim)+'\n')
            f.write('cut-off time tau: '+str(tau)+'\n\n')
            f.write(f'Train ({n*0.7}):\n')
            f.write(f'Events:     {round(np.mean(portion_events_after_cut_train)*100,2)}  %,  n={round(n*0.7*np.mean(portion_events_after_cut_train),0)}\n')
            f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_no_events_after_cut_train),0)}\n')
            f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_censored_after_cut_train),0)}\n')
            f.write(f'Test  ({n*0.3}):\n')
            f.write(f'Events:     {round(np.mean(portion_events_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_events_after_cut_test),0)}\n')
            f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_test)*100,2)} %,   n={round(n*0.3*np.mean(portion_no_events_after_cut_test),0)}\n')
            f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_censored_after_cut_test),0)}\n')


            f.write('\n\n### Evaluation: ###\n')
            f.write(f'WB C-Index IPCW: {np.mean(wb_cindex_ipcw):.4f}\n')
            f.write(f'WB MSE IPCW: {np.mean(wb_mse_ipcw):.4f}\n')
            f.write(f'RF MSE IPCW: {np.mean(rf_mse_ipcw):.4f}\n')
            f.write('\n')

            f.write('\n###Prediction Results:###\n')
            f.write(f'True Y: {99999999999999:.4f}\n')
            f.write(f'WB Y_pred: {np.mean(wb_y_pred_X_point):.4f}\n')
            f.write(f'RF Y_pred: {np.mean(rf_y_pred_X_point):.4f}\n')
            f.write('\n')



            f.write('\n\n### Simulation Parameters: ###\n')
            f.write('first_level_B for bootstrap variance estimation (B_1): '+str(B_1)+'\n')

            f.write('Data Creation Parameters:\n')
            f.write(str(params_data_creation))
            
            f.write('\nRandom Forest Parameter:\n')
            f.write(str(params_rf)+'\n')
            f.write(f'the above seeds ({seed}) are start_seed for the simulation function\n')

        print(experiment_name+'   finished')


Simulations:  10%|█         | 1/10 [00:03<00:29,  3.25s/simulation]

CPU usage: 5.1%
Memory usage: 66.4%


Simulations:  20%|██        | 2/10 [00:04<00:15,  1.93s/simulation]

CPU usage: 1.3%
Memory usage: 66.5%


Simulations: 100%|██████████| 10/10 [00:05<00:00,  1.90simulation/s]

CPU usage: 1.8%
Memory usage: 66.5%


n_train 525.0_Events 43.0 (8.15 %)_Cens 51.0 (9.75 %)_(B_RF) 100__(B_1) 200   finished
